In [8]:
# 1. Install necessary libraries
!pip install dagshub mlflow --quiet

import pandas as pd
import numpy as np
import mlflow.sklearn
import dagshub
import os
from kaggle_secrets import UserSecretsClient

# 2. Initialize DagsHub (Replace with your actual token if asked, or it will prompt you)
user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("GUGA_TOKEN")
DAGSHUB_USERNAME = user_secrets.get_secret("GUGA_USERNAME")

repo_name = "ML_assignment1" 

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{repo_name}.mlflow")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

print(" Connected to DagsHub + MLflow successfully!")



 Connected to DagsHub + MLflow successfully!


In [9]:
# 3. Load the Test Data from Kaggle's input folder
test_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
test_df = pd.read_csv(test_path)
test_ids = test_df['Id']


In [10]:
# 4. Cleaning & Encoding Logic (UPDATED to match Training File)
def prepare_test(df):
    temp = df.copy()
    
    cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
    temp = temp.drop(columns=[col for col in cols_to_drop if col in temp.columns])
    
    
    num_cols = temp.select_dtypes(include=[np.number]).columns
    temp[num_cols] = temp[num_cols].fillna(temp[num_cols].median())
    
    cat_cols = temp.select_dtypes(include=['object']).columns
    for col in cat_cols:
        temp[col] = temp[col].fillna("None")
        temp[col] = pd.factorize(temp[col])[0]
    return temp

X_test_cleaned = prepare_test(test_df)



In [11]:
# 5. Load the CHAMPION model from DagsHub Registry

model_uri = "models:/HousePriceBestModel/latest" 
loaded_model = mlflow.sklearn.load_model(model_uri)
print(f" Model loaded successfully from: {model_uri}")



 Model loaded successfully from: models:/HousePriceBestModel/latest


In [12]:
# 6. Feature Selection (USE ALL COLUMNS for the Base Linear Model)
X_test_final = X_test_cleaned.copy()

X_test_final = X_test_final[loaded_model.feature_names_in_]

print(f" Data prepared with {X_test_final.shape[1]} features to match the Base Model.")

 Data prepared with 75 features to match the Base Model.


In [13]:
# 7. Generate Predictions
predictions = loaded_model.predict(X_test_final)

# 8. Create Submission File
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": predictions
})

# Final Safety Check: Ensure no negative prices
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved to:", submission_path)
print("Files in working dir:", os.listdir("/kaggle/working"))

Saved to: /kaggle/working/submission.csv
Files in working dir: ['submission.csv', '.virtual_documents']
